# 🧠 RoleNet Training — Vietnamese Social Role Classifier

**Model:** MobileNetV3-Small (fine-tuned)  
**Classes:** 16 vai trò xã hội Việt Nam  
**Dataset:** VN-Role-16 (~27,232 ảnh sau augmentation)  
**Target:** ≥85% Top-1 Accuracy  

---
### ⚡ Yêu cầu:
- Runtime → Change runtime type → **T4 GPU**
- Upload `rolenet_upload.zip` lên Google Drive trước

## 📌 Bước 1: Mount Google Drive & Giải nén Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ⚙️ Đường dẫn file ZIP trên Google Drive (chỉnh lại nếu cần)
ZIP_PATH = '/content/drive/MyDrive/rolenet_upload.zip'
EXTRACT_DIR = '/content/rolenet_dataset'

if not os.path.exists(EXTRACT_DIR):
    print('📦 Đang giải nén dataset...')
    !unzip -q "{ZIP_PATH}" -d /content/
    print('✅ Giải nén xong!')
else:
    print('✅ Dataset đã tồn tại, bỏ qua giải nén.')

# Kiểm tra cấu trúc
!ls /content/rolenet_dataset/augmented/ | head -20

# Đếm tổng ảnh
total = sum(len(list(__import__('pathlib').Path(EXTRACT_DIR + '/augmented/' + d).glob('*.jpg')))
            for d in os.listdir(EXTRACT_DIR + '/augmented'))
print(f'\n📊 Tổng ảnh augmented: {total:,}')

## 📌 Bước 2: Cài đặt thư viện

In [ ]:
!pip install -q timm torchmetrics --upgrade
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "❌ Không có GPU"}')
print(f'CUDA: {torch.cuda.is_available()}')

## 📌 Bước 3: Cấu hình & Constants

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from collections import Counter
import json, time, copy

# ============================================================
# CONFIG
# ============================================================
CFG = {
    # Dataset
    'data_root'    : '/content/rolenet_dataset/augmented',
    'splits_dir'   : '/content/rolenet_dataset/splits',
    'img_size'     : (256, 128),      # (H, W) — person crop dọc

    # Model
    'model_name'   : 'mobilenet_v3_small',
    'num_classes'  : 16,
    'pretrained'   : True,

    # Training
    'epochs'       : 30,
    'batch_size'   : 64,
    'lr'           : 3e-4,
    'weight_decay' : 1e-4,
    'label_smooth' : 0.1,
    'warmup_epochs': 3,

    # Output
    'save_dir'     : '/content/drive/MyDrive/rolenet_checkpoints',
    'device'       : 'cuda' if torch.cuda.is_available() else 'cpu',
    'num_workers'  : 2,
    'seed'         : 42,
}

# Class mapping
CLASS_NAMES = [
    'shipper', 'doctor', 'police', 'military', 'security', 'student',
    'chef', 'janitor', 'construction', 'nurse', 'postman', 'technician',
    'worker', 'civil_guard', 'normal', 'unknown'
]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

# Set seed
torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])

# Tạo thư mục save
Path(CFG['save_dir']).mkdir(parents=True, exist_ok=True)

print('✅ Config loaded!')
print(f"Device: {CFG['device']}")
print(f"Classes: {CFG['num_classes']} → {CLASS_NAMES}")

## 📌 Bước 4: Dataset & DataLoader

In [ ]:
class RoleNetDataset(Dataset):
    """Dataset load từ thư mục augmented/ theo folder structure."""

    def __init__(self, root_dir: str, split_csv: str, transform=None):
        self.root      = Path(root_dir)
        self.transform = transform
        self.samples   = []

        df = pd.read_csv(split_csv)
        for _, row in df.iterrows():
            img_path = self.root / row['path']
            label    = CLASS_TO_IDX.get(row['label'], -1)
            if img_path.exists() and label >= 0:
                self.samples.append((str(img_path), label))

        print(f'  Loaded {len(self.samples):,} samples from {Path(split_csv).name}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label


# ============================================================
# Transforms
# ============================================================
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(CFG['img_size']),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(8),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),  # Occlusion augment
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize(CFG['img_size']),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ============================================================
# Load splits
# ============================================================
print('📂 Loading datasets...')
train_ds = RoleNetDataset(CFG['data_root'], f"{CFG['splits_dir']}/train.csv", train_transform)
val_ds   = RoleNetDataset(CFG['data_root'], f"{CFG['splits_dir']}/val.csv",   val_transform)
test_ds  = RoleNetDataset(CFG['data_root'], f"{CFG['splits_dir']}/test.csv",  val_transform)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)

print(f'\n✅ Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}')

# Kiểm tra class balance
labels = [s[1] for s in train_ds.samples]
counts = Counter(labels)
print('\n📊 Class distribution (train):')
for i, name in enumerate(CLASS_NAMES):
    bar = '█' * (counts[i] // 50)
    print(f'  {name:15s} {counts[i]:5d}  {bar}')

In [ ]:
# Hiển thị sample ảnh
fig, axes = plt.subplots(2, 8, figsize=(16, 6))
mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
std  = torch.tensor(IMAGENET_STD).view(3,1,1)

for idx, ax in enumerate(axes.flat):
    img_t, label = train_ds[idx * 100 % len(train_ds)]
    img = (img_t * std + mean).clamp(0,1).permute(1,2,0).numpy()
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[label], fontsize=8)
    ax.axis('off')

plt.suptitle('Sample Training Images', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/content/sample_images.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Sample ảnh saved → /content/sample_images.png')

## 📌 Bước 5: Định nghĩa Model (MobileNetV3 Fine-tune)

In [ ]:
def build_rolenet_model(num_classes: int = 16, pretrained: bool = True) -> nn.Module:
    """
    MobileNetV3-Small fine-tuned cho RoleNet.
    - Giữ nguyên backbone, thay Classifier head
    - Thêm Dropout 0.4 để tránh overfit
    - Output: 16 classes (social roles)
    """
    weights = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None
    model   = models.mobilenet_v3_small(weights=weights)

    # Freeze backbone trong warmup
    # (sẽ unfreeze sau warmup_epochs)

    # Thay classifier
    in_features = model.classifier[0].in_features
    model.classifier = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.Hardswish(),
        nn.Dropout(p=0.4),
        nn.Linear(256, num_classes),
    )

    return model


model = build_rolenet_model(CFG['num_classes'], CFG['pretrained'])
model = model.to(CFG['device'])

# Thống kê param
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Model: MobileNetV3-Small')
print(f'   Total params    : {total_params:,}')
print(f'   Trainable params: {trainable_params:,}')
print(f'   Model size      : ~{total_params * 4 / 1024 / 1024:.1f} MB')

# Test forward pass
with torch.no_grad():
    dummy = torch.randn(2, 3, 256, 128).to(CFG['device'])
    out   = model(dummy)
    print(f'   Output shape    : {out.shape}  (batch=2, classes=16) ✅')

## 📌 Bước 6: Loss, Optimizer, Scheduler

In [ ]:
# ============================================================
# Label Smoothing Loss
# ============================================================
criterion = nn.CrossEntropyLoss(label_smoothing=CFG['label_smooth'])

# ============================================================
# Optimizer: AdamW
# ============================================================
optimizer = optim.AdamW(
    model.parameters(),
    lr           = CFG['lr'],
    weight_decay = CFG['weight_decay'],
)

# ============================================================
# Scheduler: OneCycleLR (warmup + cosine decay)
# ============================================================
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr          = CFG['lr'],
    epochs          = CFG['epochs'],
    steps_per_epoch = len(train_loader),
    pct_start       = CFG['warmup_epochs'] / CFG['epochs'],
    anneal_strategy = 'cos',
)

print('✅ Loss    : CrossEntropyLoss (label_smoothing=0.1)')
print('✅ Optim   : AdamW (lr=3e-4, wd=1e-4)')
print('✅ Schedule: OneCycleLR (warmup 3 epochs → cosine decay 27 epochs)')

## 📌 Bước 7: Training Loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0
    correct    = 0
    n          = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * imgs.size(0)
        preds       = outputs.argmax(1)
        correct    += (preds == labels).sum().item()
        n          += imgs.size(0)

    return total_loss / n, correct / n


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct    = 0
    n          = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs      = model(imgs)
        loss         = criterion(outputs, labels)

        total_loss += loss.item() * imgs.size(0)
        preds       = outputs.argmax(1)
        correct    += (preds == labels).sum().item()
        n          += imgs.size(0)

    return total_loss / n, correct / n


print('✅ Training functions defined.')

In [ ]:
# ============================================================
# MAIN TRAINING LOOP
# ============================================================
history  = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_acc = 0.0
best_weights = None
save_path = Path(CFG['save_dir'])

print(f'🚀 Bắt đầu training {CFG["epochs"]} epochs...')
print(f'   Device   : {CFG["device"]}')
print(f'   Batch    : {CFG["batch_size"]}')
print(f'   Steps/ep : {len(train_loader)}')
print('=' * 70)

t_start = time.time()

for epoch in range(1, CFG['epochs'] + 1):
    t0 = time.time()

    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, scheduler, CFG['device'])
    val_loss,   val_acc   = eval_epoch(
        model, val_loader,   criterion, CFG['device'])

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    is_best = val_acc > best_acc
    if is_best:
        best_acc     = val_acc
        best_weights = copy.deepcopy(model.state_dict())
        torch.save({
            'epoch'      : epoch,
            'model_state': model.state_dict(),
            'optimizer'  : optimizer.state_dict(),
            'val_acc'    : val_acc,
            'classes'    : CLASS_NAMES,
        }, save_path / 'rolenet_best.pt')

    lr_now = optimizer.param_groups[0]['lr']
    elapsed = time.time() - t0
    marker = ' ⭐ BEST' if is_best else ''

    print(
        f'Ep {epoch:3d}/{CFG["epochs"]} | '
        f'Loss {train_loss:.4f}/{val_loss:.4f} | '
        f'Acc {train_acc*100:.1f}%/{val_acc*100:.1f}% | '
        f'LR {lr_now:.2e} | '
        f'{elapsed:.0f}s{marker}'
    )

    # Early stop nếu val_acc >= 92%
    if val_acc >= 0.92:
        print(f'\n🎉 Đạt {val_acc*100:.1f}% ≥ 92%, dừng early!')
        break

total_time = (time.time() - t_start) / 60
print('=' * 70)
print(f'\n✅ Training xong! {total_time:.1f} phút')
print(f'   Best Val Accuracy: {best_acc*100:.2f}%')
print(f'   Saved → {save_path}/rolenet_best.pt')

## 📌 Bước 8: Vẽ Learning Curve

In [ ]:
epochs_ran = len(history['train_acc'])
x = range(1, epochs_ran + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(x, history['train_loss'], 'b-o', label='Train Loss', markersize=4)
ax1.plot(x, history['val_loss'],   'r-o', label='Val Loss',   markersize=4)
ax1.set_title('Training & Validation Loss', fontsize=13)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

# Accuracy
ax2.plot(x, [a*100 for a in history['train_acc']], 'b-o', label='Train Acc', markersize=4)
ax2.plot(x, [a*100 for a in history['val_acc']],   'r-o', label='Val Acc',   markersize=4)
ax2.axhline(y=best_acc*100, color='gold', linestyle='--', label=f'Best: {best_acc*100:.1f}%')
ax2.set_title('Training & Validation Accuracy', fontsize=13)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(); ax2.grid(alpha=0.3)
ax2.set_ylim(0, 105)

plt.suptitle('RoleNet Training History', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/training_history.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Chart saved → /content/training_history.png')

## 📌 Bước 9: Đánh Giá Trên Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Load best model
model.load_state_dict(best_weights)
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs  = imgs.to(CFG['device'])
        preds = model(imgs).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print('📊 Classification Report:')
print(classification_report(
    all_labels, all_preds,
    target_names=CLASS_NAMES,
    digits=3
))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    cm_pct, annot=True, fmt='.0f', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    ax=ax, cbar_kws={'label': '%'}
)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title(f'Confusion Matrix — RoleNet (Best Val Acc: {best_acc*100:.1f}%)', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix saved → /content/confusion_matrix.png')

## 📌 Bước 10: Export Model (TorchScript + ONNX)

In [ ]:
model.load_state_dict(best_weights)
model.eval().cpu()

export_dir = Path(CFG['save_dir'])
dummy_input = torch.randn(1, 3, 256, 128)  # (batch, C, H, W)

# ── 1. TorchScript (dùng trong Python/C++) ──
try:
    scripted = torch.jit.trace(model, dummy_input)
    ts_path  = export_dir / 'rolenet_scripted.pt'
    scripted.save(str(ts_path))
    print(f'✅ TorchScript → {ts_path}')
except Exception as e:
    print(f'⚠️  TorchScript failed: {e}')

# ── 2. ONNX (dùng Edge/Mobile) ──
try:
    import onnx
    onnx_path = export_dir / 'rolenet.onnx'
    torch.onnx.export(
        model, dummy_input, str(onnx_path),
        input_names  = ['image'],
        output_names = ['logits'],
        dynamic_axes = {'image': {0: 'batch_size'}},
        opset_version = 12,
    )
    onnx_model = onnx.load(str(onnx_path))
    onnx.checker.check_model(onnx_model)
    size_mb = onnx_path.stat().st_size / 1024 / 1024
    print(f'✅ ONNX       → {onnx_path} ({size_mb:.1f} MB)')
except Exception as e:
    print(f'⚠️  ONNX export failed: {e}')

# ── 3. Lưu metadata ──
metadata = {
    'model'        : 'MobileNetV3-Small',
    'num_classes'  : CFG['num_classes'],
    'classes'      : CLASS_NAMES,
    'class_to_idx' : CLASS_TO_IDX,
    'input_size'   : [256, 128],
    'best_val_acc' : round(best_acc * 100, 2),
    'imagenet_mean': IMAGENET_MEAN,
    'imagenet_std' : IMAGENET_STD,
    'label_map'    : {str(i): c for i, c in enumerate(CLASS_NAMES)},
}
meta_path = export_dir / 'rolenet_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'✅ Metadata   → {meta_path}')
print(f'\n🎉 Export hoàn tất! Tất cả files trong: {export_dir}')
print('\n📥 Files cần download về máy:')
for f in export_dir.iterdir():
    sz = f.stat().st_size / 1024
    print(f'   {f.name:30s} {sz:8.1f} KB')

## 📌 Bước 11: Test Nhanh Inference

In [ ]:
import time

model.eval().to(CFG['device'])

def predict_role(img_path: str) -> tuple[str, float]:
    """Dự đoán vai trò từ 1 ảnh."""
    img = Image.open(img_path).convert('RGB')
    x   = val_transform(img).unsqueeze(0).to(CFG['device'])
    with torch.no_grad():
        logits = model(x)
        probs  = torch.softmax(logits, dim=1)[0]
    top_idx  = probs.argmax().item()
    top_conf = probs[top_idx].item()
    return CLASS_NAMES[top_idx], top_conf, probs.cpu().numpy()


# Test trên 8 ảnh ngẫu nhiên từ test set
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
mean_t = torch.tensor(IMAGENET_MEAN).view(3,1,1)
std_t  = torch.tensor(IMAGENET_STD).view(3,1,1)

for ax in axes.flat:
    idx          = np.random.randint(len(test_ds))
    path, gt_lbl = test_ds.samples[idx]
    pred_cls, conf, _ = predict_role(path)
    gt_cls       = CLASS_NAMES[gt_lbl]

    img_t, _ = test_ds[idx]
    img = (img_t * std_t + mean_t).clamp(0,1).permute(1,2,0).numpy()
    ax.imshow(img)

    color = 'green' if pred_cls == gt_cls else 'red'
    ax.set_title(f'GT: {gt_cls}\nPred: {pred_cls} ({conf*100:.0f}%)',
                 color=color, fontsize=9)
    ax.axis('off')

plt.suptitle('Inference Demo — RoleNet', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/inference_demo.png', dpi=120, bbox_inches='tight')
plt.show()

# Benchmark speed
model.eval()
dummy = torch.randn(1, 3, 256, 128).to(CFG['device'])
# Warmup
for _ in range(10):
    with torch.no_grad(): model(dummy)
# Measure
t0 = time.perf_counter()
for _ in range(100):
    with torch.no_grad(): model(dummy)
ms_per_frame = (time.perf_counter() - t0) / 100 * 1000
print(f'\n⚡ Inference speed: {ms_per_frame:.2f} ms/frame ({1000/ms_per_frame:.0f} FPS) on {CFG["device"].upper()}')

## 📌 Bước 12: Download Files Về Máy

In [ ]:
from google.colab import files
import shutil

# Copy charts về Drive
for f in ['training_history.png', 'confusion_matrix.png', 'inference_demo.png']:
    src = f'/content/{f}'
    dst = str(save_path / f)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'✅ Copied {f} → Drive')

print('\n📥 Download files về máy...')

# Download model files
for fname in ['rolenet_best.pt', 'rolenet_scripted.pt', 'rolenet_metadata.json']:
    fpath = Path(CFG['save_dir']) / fname
    if fpath.exists():
        print(f'   Downloading {fname}...')
        files.download(str(fpath))

print('\n✅ Tất cả đã download! Lưu vào thư mục models/ trong project.')

---
## 📝 Ghi Chú Tích Hợp

Sau khi có `rolenet_best.pt`, cập nhật `modules/role_classifier.py`:

```python
# Trong RoleClassifier.__init__:
import torch
from torchvision import models

checkpoint = torch.load('models/rolenet_best.pt', map_location='cpu')
model = models.mobilenet_v3_small(weights=None)
model.classifier = nn.Sequential(
    nn.Linear(576, 256), nn.Hardswish(), nn.Dropout(0.4),
    nn.Linear(256, 16)
)
model.load_state_dict(checkpoint['model_state'])
model.eval()
self.rolenet = model

# Trong classify():
# crop person → resize 128×256 → normalize → model.forward() → argmax
```

**Files download về:**
- `rolenet_best.pt` → `C:/code/camera-ai/models/rolenet_best.pt`
- `rolenet_metadata.json` → `C:/code/camera-ai/models/rolenet_metadata.json`